In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('s3://cs480-injury-mortality-data/injury_mortality_final_cleaned (1).csv')

#Filter out summary rows for training
train_df = df[df['Is_Summary_Row'] == 0].copy()

# Simple Encoding (Target_Rate is our label)
cols = ['Target_Rate', 'Year', 'Sex', 'Race', 'Injury mechanism']
data = pd.get_dummies(train_df[cols]) # Converts text to numbers

# 4. Split and Save
train, test = train_test_split(data, test_size=0.2, random_state=42)
train.to_csv('train.csv', header=False, index=False)
test.to_csv('test.csv', header=False, index=False)

In [2]:
import boto3
import os

# Define my bucket and folder
bucket_name = 'cs480-injury-mortality-data'
prefix = 'Final_Data'

# Upload train.csv
boto3.Session().resource('s3').Bucket(bucket_name).Object(
    os.path.join(prefix, 'train/train.csv')).upload_file('train.csv')

# Upload test.csv
boto3.Session().resource('s3').Bucket(bucket_name).Object(
    os.path.join(prefix, 'test/test.csv')).upload_file('test.csv')

print("Files successfully uploaded to S3!")

Files successfully uploaded to S3!


In [4]:
import sagemaker
import boto3
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

# 1. Setup
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = 'cs480-injury-mortality-data'

# 2. Get the XGBoost Container Image
container = image_uris.retrieve("xgboost", sess.boto_region_name, "1.5-1")

# 3. Point to folders (Make sure the CSVs are inside these exact folders in S3)
s3_input_train = TrainingInput(s3_data=f's3://{bucket}/Final_Data/train/', content_type='csv')
s3_input_test = TrainingInput(s3_data=f's3://{bucket}/Final_Data/test/', content_type='csv')

# 4. Define the Estimator
xgb = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/sagemaker/output',
    sagemaker_session=sess
)

# 5. Set Hyperparameters
xgb.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
    objective='reg:squarederror',
    num_round=50
)

# 6. RUN TRAINING
xgb.fit({'train': s3_input_train, 'validation': s3_input_test})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-04-06 01:39:00 Starting - Starting the training job...
2026-04-06 01:39:14 Starting - Preparing the instances for training...
2026-04-06 01:39:36 Downloading - Downloading input data...
2026-04-06 01:40:22 Downloading - Downloading the training image......
2026-04-06 01:41:28 Training - Training image download completed. Training in progress.
2026-04-06 01:41:28 Uploading - Uploading generated training model/miniconda3/lib/python3.8/site-packages/xgbo

In [5]:
# Create a model object from the training job you just finished
model = xgb.create_model(role=role)

# Register the model version
model.register(
    model_package_group_name="MortalityPredictionGroup",
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium"],
    transform_instances=["ml.m5.large"],
    description="V1: XGBoost model for injury mortality rates",
    approval_status="PendingManualApproval"
)
print("Model successfully registered!")

Model successfully registered!
